In [13]:
%pip install isaacus python-dotenv requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from isaacus import Isaacus
from dotenv import load_dotenv
import os
import requests
from examples import EXAMPLE_DOCUMENTS
from custom_taxonomy import CUSTOM_TAXONOMY

In [16]:
load_dotenv()  # Load environment variables from `.env` files if any are present.
isaacus_client = Isaacus(api_key=os.getenv("ISAACUS_API_KEY"))

TAXONOMY

TITLE | CLIENT | DATE 

In [17]:
def clean_text(text):
    return text.replace("\n", " ").strip() if text else None

def extract_date(doc):
    for date in doc.dates:
        if date.type == "creation":
            return date.value
    return None

def extract_title(doc):
    return clean_text(doc.title.decode(doc.text)) if doc.title else None

def extract_parties(doc):
    parties = []
    for entity in doc.persons:
        if entity.parent is None:
            entity_name = clean_text(entity.name.decode(doc.text))
            parties.append(entity_name)

    return parties
def extract_locations(doc):
    locations = []
    for entity in doc.locations:
        entity_name = clean_text(entity.name.decode(doc.text))
        locations.append(entity_name)

    return locations
def extract_terms(doc):
    terms = []
    for term in doc.terms:
        term_name = clean_text(term.name.decode(doc.text))
        term_definition = clean_text(term.meaning.decode(doc.text))
        terms.append({"name": term_name, "definition": term_definition})

    return terms

def extradct_signatures(doc):
    signatures = []
    for segment in doc.segments:
        if segment.type == "signature":
            signatures.append(segment.span.decode(doc.text))
    return signatures

def extract_enriched_features_from_document(document_text):
    response = isaacus_client.enrichments.create(
        model="kanon-2-enricher",
        texts=[document_text],
        overflow_strategy="auto",
    )
    doc = response.results[0].document
    features = {}
    features["title"] = extract_title(doc)
    features["parties"] = extract_parties(doc)
    features["date"] = extract_date(doc)
    features["locations"] = extract_locations(doc)
    features["terms"] = extract_terms(doc)
    features["signatures"] = extradct_signatures(doc)
    return features


In [18]:

def score_document_by_category(document_text, query):
    result = isaacus_client.rerankings.create(
        model="kanon-2-reranker",
        query=query,
        texts=[document_text],
    )
    return result.results[0].score

def classify_document(document, nodes, mode="full"):
    if mode == "greedy":
        return classify_greedy(document, nodes)
    if mode == "full":
        return classify_full(document, nodes)

def classify_greedy(document: str, nodes: list[dict]) -> dict:
    scored_nodes = [{
            "node": node,
            "score": score_document_by_category(document, node["description"])
        }
        for node in nodes
    ]

    best = max(scored_nodes, key=lambda item: item["score"])
    best_node = best["node"]

    if not best_node.get("children"):
        return {
            "label": best_node["name"],
            "score": best["score"],
            "description": best_node["description"],
            "mode": "greedy",
        }

    return classify_greedy(document, best_node["children"])


def flatten_taxonomy(nodes):
    all_nodes = []

    for node in nodes:
        all_nodes.append(node)
        children = node.get("children", [])
        if children:
            all_nodes.extend(flatten_taxonomy(children))

    return all_nodes


def classify_full(document, nodes):
    all_nodes = flatten_taxonomy(nodes)

    scored_nodes = [
        {
            "node": node,
            "score": score_document_by_category(document, node["description"]),
        }
        for node in all_nodes
    ]

    best = max(scored_nodes, key=lambda item: item["score"])
    best_node = best["node"]

    return {
        "label": best_node["name"],
        "score": best["score"],
        "description": best_node["description"],
        "mode": "full",
    }



In [19]:
def pretty_title(label, parties, date):
    year = date.split("-")[0] if date else None
    if parties and date:
        pretty_title = f"{label} - {parties[0]} ({year})"
    elif parties:
        pretty_title = f"{label} - {parties[0]}"
    elif date:
        pretty_title= f"{label} ({year[:-2]})"
    return pretty_title


In [20]:
def extract_metadata_from_text(document_text, taxonomy_mode="full"):
    features = extract_enriched_features_from_document(document_text)
    category = classify_document(document_text, CUSTOM_TAXONOMY, mode=taxonomy_mode)
    features["category"] = category
    features["pretty_title"] = pretty_title(category["label"], features["parties"], features["date"])
    print(features)


In [21]:
import base64
import json
import ipywidgets as widgets
from IPython.display import HTML, display

# Replace these with your real notebook functions.
# They should return only the fields you already extract.
def __metadata_viewer_process(payload):
    kind = payload.get("kind")
    name = payload.get("name", "Untitled")

    if kind == "text":
        document_text = payload["text"]
    elif kind == "file":
        raw_bytes = base64.b64decode(payload["base64"])
        try:
            document_text = raw_bytes.decode("utf-8")
        except UnicodeDecodeError:
            document_text = raw_bytes.decode("latin-1", errors="ignore")
    else:
        raise ValueError(f"Unsupported payload kind: {kind}")

    # Plug in your existing notebook logic here
    features = extract_enriched_features_from_document(document_text)
    category = classify_document(document_text, CUSTOM_TAXONOMY, mode="full")

    features["category"] = category
    features["source_name"] = name
    return features

# Hidden widget bridge used by the HTML
request_box = widgets.Textarea(layout=widgets.Layout(display="none"))
response_box = widgets.Textarea(layout=widgets.Layout(display="none"))
submit_button = widgets.Button(description="submit", layout=widgets.Layout(display="none"))

request_box.add_class("metadata_viewer_request")
response_box.add_class("metadata_viewer_response")
submit_button.add_class("metadata_viewer_submit")

def _handle_submit(_):
    try:
        payload = json.loads(request_box.value)
        result = __metadata_viewer_process(payload)
        response_box.value = json.dumps(result)
    except Exception as e:
        response_box.value = json.dumps({"error": str(e)})

submit_button.on_click(_handle_submit)

# Give the widgets fixed DOM ids for the HTML bridge
display(HTML("""
<div id="bridge-mount"></div>
<script>
setTimeout(() => {
  const req = document.querySelector(".metadata_viewer_request textarea, .metadata_viewer_request");
  const res = document.querySelector(".metadata_viewer_response textarea, .metadata_viewer_response");
  const btn = document.querySelector(".metadata_viewer_submit button, .metadata_viewer_submit");
  if (req) req.id = "metadata_viewer_request";
  if (res) res.id = "metadata_viewer_response";
  if (btn) btn.id = "metadata_viewer_submit";
}, 300);
</script>
"""))

display(request_box, response_box, submit_button)

# Render the viewer
display(HTML(open("metadata_viewer_widgets.html", "r", encoding="utf-8").read()))

Textarea(value='', layout=Layout(display='none'), _dom_classes=('metadata_viewer_request',))

Textarea(value='', layout=Layout(display='none'), _dom_classes=('metadata_viewer_response',))

Button(description='submit', layout=Layout(display='none'), style=ButtonStyle(), _dom_classes=('metadata_viewe…